# state-dict-load — faded example 1: Call load_state_dict with strict=False

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `state-dict-load`. Running the beacon reports progress on the `Transfer: state_dict load` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Transfer: state_dict load` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`state-dict-load`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "state-dict-load"
DD_SUBTOPIC = "Transfer: state_dict load"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Passing `strict=False` makes `load_state_dict` tolerate key mismatches and return an `_IncompatibleKeys(missing_keys, unexpected_keys)` tuple instead of raising. This is required when the checkpoint and model do not share every key.

## Faded exercise 1

Complete `load_partial(model, ckpt)` to load the checkpoint non-strictly and return `(missing, unexpected)` as Python lists. Fill in the load call that uses `strict=False`.

**Fill in:** call model.load_state_dict with strict=False and capture the result

In [ ]:
import torch.nn as nn


class ToyNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.backbone = nn.Linear(10, 16)
        self.fc = nn.Linear(16, n_classes)
    def forward(self, x):
        return self.fc(self.backbone(x))


def load_partial(model, ckpt):
    result = None  # TODO: call model.load_state_dict with strict=False and capture the result
    return list(result.missing_keys), list(result.unexpected_keys)


t.manual_seed(0)
model = ToyNet(7)
ckpt = {'backbone.weight': t.randn(16, 10), 'backbone.bias': t.randn(16), 'extra': t.randn(3)}
missing, unexpected = load_partial(model, ckpt)

def _test():
    t.manual_seed(0)
    model = ToyNet(7)
    ckpt = {'backbone.weight': t.randn(16, 10), 'backbone.bias': t.randn(16), 'extra': t.randn(3)}
    missing, unexpected = load_partial(model, ckpt)
    assert sorted(missing) == ['fc.bias', 'fc.weight'], f'missing wrong: {missing}'
    assert sorted(unexpected) == ['extra'], f'unexpected wrong: {unexpected}'

try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn


class ToyNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.backbone = nn.Linear(10, 16)
        self.fc = nn.Linear(16, n_classes)
    def forward(self, x):
        return self.fc(self.backbone(x))


def load_partial(model, ckpt):
    result = model.load_state_dict(ckpt, strict=False)
    return list(result.missing_keys), list(result.unexpected_keys)


t.manual_seed(0)
model = ToyNet(7)
ckpt = {'backbone.weight': t.randn(16, 10), 'backbone.bias': t.randn(16), 'extra': t.randn(3)}
missing, unexpected = load_partial(model, ckpt)
```
</details>